In [8]:
!pip install pinecone


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [44]:
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
import time
import pandas as pd 
import os
import dotenv
dotenv.load_dotenv()


True

In [24]:
token = os.getenv('RUNPOD_API_KEY')
open_ai_base_url = os.getenv("RUNPOD_EMBEDDING_URL")
model_name = os.getenv("MODEL_NAME")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX_NAME")

print(pinecone_api_key)
print(token)
print(open_ai_base_url)

pcsk_2xh8Vd_M37ohMu8CEMr1HmCd31jxvNeSisLfg3XU9SA4E1rPa43Jh8DmQfM2pHfLg2WS5Z
rpa_JZXH5FSS9RK981F4COH1FFYHVEO7LRSPVA0WNA5O1fas1a
https://api.runpod.ai/v2/0hrivws5y993bs/openai/v1


In [25]:
pc = Pinecone(api_key=pinecone_api_key)
client = OpenAI(
    api_key=token,
    base_url=open_ai_base_url
)

# Try Out Embeddings


In [26]:
output = client.embeddings.create(input=["hello world"], model=model_name)
embedding = output.data[0].embedding
print(embedding)

[0.015215358696877956, -0.02272770367562771, 0.008572462946176529, -0.07437602430582047, 0.003935400862246752, 0.0027780423406511545, -0.03130016475915909, 0.0446622259914875, 0.04399107024073601, -0.007783094421029091, -0.02524453029036522, -0.033374641090631485, 0.014376416802406311, 0.046340107917785645, 0.00868686381727457, -0.0160466730594635, 0.007504718378186226, -0.019005851820111275, -0.11470626294612885, -0.01813640259206295, 0.1262989193201065, 0.029729057103395462, 0.025229275226593018, -0.0341678224503994, -0.04109290614724159, 0.006604762282222509, 0.010349494405090809, 0.02239212580025196, 0.004431139677762985, -0.12776325643062592, -0.016061928123235703, -0.020348157733678818, 0.04737734794616699, 0.011585026979446411, 0.06827463209629059, 0.007413197308778763, -0.018044881522655487, 0.040970880538225174, -0.010196959599852562, 0.02370392717421055, 0.010410508140921593, -0.02846301719546318, 0.008091977797448635, -0.015253492631018162, 0.03090357594192028, -0.0659561008

In [27]:
len(embedding)

384

# Wrangle dataset 

In [29]:
df = pd.read_json("products/products.jsonl", lines=True)
df.head(2)

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp


In [30]:
df['text'] = df['name']+" : "+df["description"]+\
    " -- Ingredients: "+df["ingredients"].astype(str) +\
    " -- Price: "+df["price"].astype(str) +\
    "-- rating: "+df["ingredients"].astype(str) 

In [31]:
df['text'].head(2)

0    Cappuccino : A rich and creamy cappuccino made...
1    Jumbo Savory Scone : Deliciously flaky and but...
Name: text, dtype: object

In [32]:
texts = df['text'].to_list()

In [33]:
with open("products/Evergreen_brew_about_us.txt") as f:
    Evergreen_brew_about_section = f.read()

Evergreen_brew_about_section = "Coffee shop Evergreen Brew about section: "+Evergreen_brew_about_section
texts.append(Evergreen_brew_about_section)

In [34]:
with open("products/menu_items_text.txt") as f:
    menu_items_text = f.read()

menu_items_text = "Menu Items: "+menu_items_text
texts.append(menu_items_text)

In [35]:
menu_items_text

'Menu Items: Menu Items\n\nCappuccino – $5.25\nJumbo Savory Scone – $3.75\nLatte – $5.50\nChocolate Chip Biscotti – $2.75\nEspresso Shot – $2.50\nHazelnut Biscotti – $3.00\nChocolate Croissant – $4.25\nDark Chocolate (Drinking Chocolate) – $5.50\nCranberry Scone – $3.75\nCroissant – $3.50\nAlmond Croissant – $4.50\nGinger Biscotti – $2.75\nOatmeal Scone – $3.50\nGinger Scone – $3.75\nChocolate Syrup – $1.75\nHazelnut Syrup – $1.75\nCaramel Syrup – $1.75\nSugar-Free Vanilla Syrup – $1.75\nDark Chocolate (Packaged Chocolate) – $3.50'

# Generate Embedding 

In [36]:
output = client.embeddings.create(input=texts, model=model_name)

In [37]:
embeddings = output.data

# Push data to database 


In [43]:
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

{
    "name": "coffeshop",
    "metric": "cosine",
    "host": "coffeshop-3tqah4x.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [47]:
# wait for the index to be ready
while not pc.describe_index(index_name).status.ready:
    time.sleep(1)

index = pc.Index(index_name)

vectors = []
for text,e in zip(texts,embeddings):
    entry_id = text.split(":")[0]
    vectors.append({
        "id": entry_id,
        "values":e.embedding,
        "metadata": {"text": text}
    })

index.upsert(vectors=vectors,namespace='ns1')

{'upserted_count': 20}

# Get closest documents 

In [48]:
output = client.embeddings.create(input=["Is Cappuccino lactose-free?"],model=model_name)
embedding = output.data[0].embedding

In [49]:
results = index.query(
    namespace='ns1',
    vector=embedding,
    top_k=3,
    include_values=False,
    include_metadata=True
)

In [50]:
results

{'matches': [{'id': 'Cappuccino ',
              'metadata': {'text': 'Cappuccino : A rich and creamy cappuccino '
                                   'made with freshly brewed espresso, steamed '
                                   'milk, and a frothy milk cap. This '
                                   'delightful drink offers a perfect balance '
                                   'of bold coffee flavor and smooth milk, '
                                   'making it an ideal companion for relaxing '
                                   'mornings or lively conversations. -- '
                                   "Ingredients: ['Espresso', 'Steamed Milk', "
                                   "'Milk Foam'] -- Price: 4.5-- rating: "
                                   "['Espresso', 'Steamed Milk', 'Milk Foam']"},
              'score': 0.727169514,
              'values': []},
             {'id': 'Sugar Free Vanilla syrup ',
              'metadata': {'text': 'Sugar Free Vanilla syrup : Enjoy t